# EEG · 04 · Generation with frozen SD (Experiment 4)
**Question:** can the EEG-predicted representations guide a reasonable image?

Stable Diffusion, CLIP and the VAE stay **frozen**; only the small token adapter is trained. Needs a GPU and the diffusers weights.

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config, load_json, get_experiment_paths
from src.data import build_datamodule
from src.generation import generate_images, train_token_adapter
cfg = load_config('configs/EEG/exp04_generation.yaml')
dm = build_datamodule(cfg).prepare()
print('experiment:', cfg['experiment']['name'], '| mode:', cfg['generation']['mode'])

## (Optional) train the token adapter — the only trainable module

In [ ]:
TRAIN_ADAPTER = True
adapter_ckpt = None
if TRAIN_ADAPTER:
    cfg['generation']['adapter_epochs'] = 20
    info = train_token_adapter(cfg, dm, resume='auto')
    adapter_ckpt = info['adapter_checkpoint']
    print('adapter checkpoint:', adapter_ckpt)

## Generate for the correct condition (and the controls)

In [ ]:
decoder = 'outputs/exp03_eeg_lowlevel_multitask/checkpoints/best.pt'
outputs = generate_images(cfg, decoder, adapter_checkpoint=adapter_ckpt, split='test')
len(outputs['correct'])

## Real vs generated (correct condition)

In [ ]:
%matplotlib inline

In [ ]:
n = min(6, len(outputs['image_ids']))
fig, axes = plt.subplots(2, n, figsize=(2.4*n, 5))
for j in range(n):
    axes[0,j].imshow(outputs['real'][j]); axes[0,j].axis('off')
    axes[1,j].imshow(outputs['correct'][j]); axes[1,j].axis('off')
axes[0,0].set_ylabel('real'); axes[1,0].set_ylabel('generated')
plt.suptitle('Top: real stimulus — Bottom: brain-guided generation'); plt.tight_layout(); plt.show()

In [ ]:
load_json(get_experiment_paths(cfg, ensure=False).metadata / 'generation_params.json')

**Takeaway:** generation is exploratory here; the quantitative claim comes from the ablation comparison in notebook 05. Note: the adapter training loss is a noisy proxy for generation quality — use notebook 06 to pick the checkpoint by CLIP similarity.